In [ ]:
import re

In [ ]:
reasoning_start = "<reasoning>"
reasoning_end = "</reasoning>"
solution_start = "<answer>"
solution_end = "</answer>"


SYSTEM_PROMPT = f"""You are a helpful assistant"""

TEMPLATE = """<start_of_turn>user
{system_prompt}

{question}<end_of_turn>
<start_of_turn>model"""

In [ ]:
# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def extract_between_tags(text: str, tag_start: str, tag_end: str) -> Optional[str]:
    """Extract content between XML-style tags"""
    pattern = rf"{re.escape(tag_start)}(.*?){re.escape(tag_end)}"
    match = re.search(pattern, text, flags=re.DOTALL | re.MULTILINE)
    return match.group(1).strip() if match else None


def extract_reasoning(output: str) -> Optional[str]:
    """Extract reasoning section"""
    return extract_between_tags(output, reasoning_start, reasoning_end)


def extract_answer(output: str) -> Optional[str]:
    """Extract answer section"""
    return extract_between_tags(output, solution_start, solution_end)

## In this notebook we will list a collection of reward functions that we can use in the GRPO

#### 1. Format Accuracy

- This is the basic reward that we can give the model for creating the answer in the required tags

In [ ]:
match_format = re.compile(
    rf"^[\s]{{0,}}"
    rf"{reasoning_start}.+?{reasoning_end}.*?"
    rf"{solution_start}(.+?){solution_end}"
    rf"[\s]{{0,}}$",
    flags=re.MULTILINE | re.DOTALL,
)

match_format.search(
    f"{reasoning_start}Let me"
    f" think!{reasoning_end}{solution_start}2{solution_end}",
)


def match_format_exactly(prompts, completions, **kwargs):
    return [
        0 if match_format.search(response) is None else 3.0
        for response in completions
    ]


def match_format_approximately(prompts, completions, **kwargs):
    scores = []

    for completion in completions:
        score = 0
        response = completion
        # Count how many keywords are seen - we penalize if too many!
        # If we see 1, then plus some points!
        score += 0.5 if response.count(reasoning_start) == 1 else -0.5
        score += 0.5 if response.count(reasoning_end) == 1 else -0.5
        score += 0.5 if response.count(solution_start) == 1 else -0.5
        score += 0.5 if response.count(solution_end) == 1 else -0.5
        scores.append(score)
    return scores


def format_content_check(prompt: str, completions: str,**kwargs) -> float:
    """
    Check that sections contain actual content, not just tags.
    
    Returns:
        Proportion of sections that have meaningful content
    """
    reasoning = extract_reasoning(completions)
    answer = extract_answer(completions)
    
    score = 0.0
    
    # Reasoning should have at least 20 characters
    if reasoning and len(reasoning.strip()) >= 20:
        score += 0.5
    
    # Answer should have at least 1 character
    if answer and len(answer.strip()) >= 1:
        score += 0.5
    
    return score

2. Reasoning Length & Depth

In [ ]:
def reasoning_length_reward(prompt: str, completions: str,**kwargs) -> float:
    """
    Reward appropriate reasoning length.
    
    Returns:
        1.0 if reasoning is within target range
        Scaled reward otherwise
    """
    reasoning = extract_reasoning(completions)
    if not reasoning:
        return 0.0
    
    # Target: 100-2000 tokens (rough estimate: 1 token ≈ 4 chars)
    target_min_chars = 400
    target_max_chars = 8000
    
    length = len(reasoning)
    
    if target_min_chars <= length <= target_max_chars:
        return 1.0
    elif length < target_min_chars:
        return max(0.0, length / target_min_chars)
    else:  # length > target_max
        # Gentle penalty for being too long
        return max(0.3, 1.0 - (length - target_max_chars) / target_max_chars)


def reasoning_depth_reward(prompt: str, completions: str,**kwargs) -> float:
    """
    Reward substantive reasoning with explicit steps.
    
    Returns:
        Score based on presence of reasoning indicators
    """
    reasoning = extract_reasoning(completions)
    if not reasoning:
        return 0.0
    
    # Count reasoning indicators
    indicators = [
        'because', 'therefore', 'thus', 'since', 'given that', 'hence',
        'first', 'second', 'third', 'next', 'then', 'finally',
        'step 1', 'step 2', 'step 3',
        'we need to', 'let\'s', 'we can', 'we should',
        'consider', 'note that', 'observe', 'recall',
        'this means', 'which implies', 'it follows that'
    ]
    
    reasoning_lower = reasoning.lower()
    indicator_count = sum(1 for ind in indicators if ind in reasoning_lower)
    
    # Normalize by length
    words = len(reasoning.split())
    if words == 0:
        return 0.0
    
    # Target: ~3-5 indicators per 100 words
    density = indicator_count / (words / 100)
    
    # Reward curve: peak at 4 indicators per 100 words
    if density < 4:
        return min(1.0, density / 4.0)
    else:
        return min(1.0, 1.0 - (density - 4) / 8.0)


def reasoning_structure_reward(prompt: str, completions: str,**kwargs) -> float:
    """
    Reward structured reasoning (numbered steps, clear progression).
    
    Returns:
        Score based on structural elements
    """
    reasoning = extract_reasoning(completions)
    if not reasoning:
        return 0.0
    
    score = 0.0
    
    # Check for numbered steps
    numbered_steps = len(re.findall(r'\n\d+[\.)]\s+', reasoning))
    if numbered_steps >= 2:
        score += 0.4
    
    # Check for bullet points or dashes
    bullet_points = len(re.findall(r'\n\s*[-•*]\s+', reasoning))
    if bullet_points >= 2:
        score += 0.3
    
    # Check for newlines (paragraphing)
    newlines = reasoning.count('\n')
    if newlines >= 2:
        score += 0.3
    
    return min(1.0, score)